# Top ddE double-mutation pairs: dE / ddE table

Reads the top-20 drug-resistance double-mutation pairs for IN, PR and RT
(`{protein}/data/top_20_DRM.txt`) and computes, **relative to the consensus
sequence**, for every pair:

| column | meaning |
|---|---|
| `dE_m1`  | delta E of the first single mutation on the consensus background |
| `dE_m2`  | delta E of the second single mutation on the consensus background |
| `dE_dbl` | delta E of the double mutation on the consensus background |
| `ddE`    | `dE_dbl - dE_m1 - dE_m2` (epistasis) |

All energies come from `utilities.functions.calculate_dde_v2`
(= `calculate_dde_from_de_with_substitution`), evaluated on the reduced-alphabet
consensus sequence with the Potts couplings `J`.

Note: RT's `top_20_DRM.txt` holds 40 pairs — the first 20 are NRTI, the last 20
NNRTI — so RT rows carry a `drug_class` label.

In [1]:
import sys
import importlib
sys.path.append('../')  # repo root

import pandas as pd
import utilities.functions as functions
importlib.reload(functions)

<module 'utilities.functions' from '/Users/xuechenkan/potts_model_test/ms0_5/../utilities/functions.py'>

## Load consensus sequences, reduction dictionaries and J matrices

In [2]:
IN_consensus_seq = open('IN/data/in.consensus.reduce4.seq').read().strip()
PR_consensus_seq = open('PR/data/pr.consensus.reduce4.seq').read().strip()
RT_consensus_seq = open('RT/data/rt.consensus.reduce4.seq').read().strip()

IN_min_position, IN_max_position = 1, 263
PR_min_position, PR_max_position = 1, 99
RT_min_position, RT_max_position = 39, 226

IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux', 1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux', 0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux', 0)

IN_J = functions.load_J_dict('IN/data/J.npy',    IN_min_position, IN_max_position)
PR_J = functions.load_J_dict('PR/data/J_PR.npy', PR_min_position, PR_max_position)
RT_J = functions.load_J_dict('RT/data/J_RT.npy', RT_min_position, RT_max_position)

proteins = {
    'IN': dict(pairs_file='IN/data/top_20_DRM.txt', consensus=IN_consensus_seq,
               redux=IN_redux, J=IN_J, min_pos=IN_min_position, max_pos=IN_max_position),
    'PR': dict(pairs_file='PR/data/top_20_DRM.txt', consensus=PR_consensus_seq,
               redux=PR_redux, J=PR_J, min_pos=PR_min_position, max_pos=PR_max_position),
    'RT': dict(pairs_file='RT/data/top_20_DRM.txt', consensus=RT_consensus_seq,
               redux=RT_redux, J=RT_J, min_pos=RT_min_position, max_pos=RT_max_position),
}

## Compute dE_m1, dE_m2, dE_dbl and ddE for every pair

In [3]:
# RT's file is NRTI (first 20) followed by NNRTI (last 20)
RT_N_NRTI = 20


def drug_class(protein, idx):
    if protein != 'RT':
        return {'IN': 'INSTI', 'PR': 'PI'}[protein]
    return 'NRTI' if idx < RT_N_NRTI else 'NNRTI'


def dde_table(protein, cfg):
    rows = []
    pairs = functions.read_list_from_file(cfg['pairs_file'])
    for idx, pair in enumerate(pairs):
        mut1, mut2 = [m.strip() for m in pair.split('-')]
        red1 = functions.unreduced_to_reduced(cfg['redux'], mut1)
        red2 = functions.unreduced_to_reduced(cfg['redux'], mut2)
        if '-' in (red1[0], red1[-1], red2[0], red2[-1]):
            print(f'[{protein}] skipped {mut1}-{mut2}: not in reduction dictionary')
            continue
        dE_m1, dE_m2, dE_dbl, ddE = functions.calculate_dde_v2(
            red1, red2, cfg['consensus'], cfg['J'], cfg['min_pos'], cfg['max_pos'])
        rows.append({
            'protein': protein,
            'drug_class': drug_class(protein, idx),
            'pair': f'{mut1}-{mut2}',
            'mut1': mut1,
            'mut2': mut2,
            'mut1_reduced': red1,
            'mut2_reduced': red2,
            'dE_m1': dE_m1,
            'dE_m2': dE_m2,
            'dE_dbl': dE_dbl,
            'ddE': ddE,
        })
    return pd.DataFrame(rows)


tables = {p: dde_table(p, cfg) for p, cfg in proteins.items()}
df = pd.concat(tables.values(), ignore_index=True)
df

,protein,drug_class,pair,mut1,mut2,mut1_reduced,mut2_reduced,dE_m1,dE_m2,dE_dbl,ddE
0,IN,INSTI,G140S-Q148H,G140S,Q148H,C140D,D148B,-5.728679,-4.553505,-1.773399,8.508784
1,IN,INSTI,Y143C-S230R,Y143C,S230R,D143A,D230C,-6.367994,-5.990549,-5.981004,6.377539
2,IN,INSTI,G140A-Q148K,G140A,Q148K,C140A,D148A,-6.462065,-7.125413,-7.974494,5.612984
3,IN,INSTI,G140S-Q148R,G140S,Q148R,C140D,D148C,-5.728679,-4.153613,-4.678102,5.204190
4,IN,INSTI,G140A-Q148R,G140A,Q148R,C140A,D148C,-6.462065,-4.153613,-5.700738,4.914940
...,...,...,...,...,...,...,...,...,...,...,...
75,RT,NNRTI,V108I-V189I,V108I,V189I,B108C,D189A,-4.285306,-5.001471,-8.112434,1.174342
76,RT,NNRTI,K101E-E138K,K101E,E138K,C101A,C138A,-3.861675,-5.320679,-8.030475,1.151880
77,RT,NNRTI,E138A-G190E,E138A,G190E,C138B,D190A,-3.707515,-5.147928,-7.760716,1.094728
78,RT,NNRTI,K101P-D192N,K101P,D192N,C101D,A192D,-5.446384,-6.151916,-10.533675,1.064624


## Write the CSVs

In [4]:
for protein, tbl in tables.items():
    out = f'{protein}_top_20_DRM_dde.csv'
    tbl.to_csv(out, index=False)
    print(f'wrote {out}  ({len(tbl)} pairs)')

df.to_csv('top_20_DRM_dde_all_proteins.csv', index=False)
print(f'wrote top_20_DRM_dde_all_proteins.csv  ({len(df)} pairs)')

wrote IN_top_20_DRM_dde.csv  (20 pairs)
wrote PR_top_20_DRM_dde.csv  (20 pairs)
wrote RT_top_20_DRM_dde.csv  (40 pairs)
wrote top_20_DRM_dde_all_proteins.csv  (80 pairs)
